# Lecture 6 · Notebook 2 — GRPO, baselines and pass@k on a tiny verifiable task
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NLP-Course-CUHKSZ/NLP-course-cuhksz.github.io/blob/main/materials/tutorials/lecture06/lecture06_2_grpo_and_pass_at_k.ipynb)

**Companion to [Chapters 4–5 of the video lecture](https://nlp-course-cuhksz.github.io/lecture06.html#p5-s58)** (slides 41–60 and 75–76). All the critic-free RL algorithms in the
lecture are REINFORCE plus a **baseline**: ReMax uses the reward of the greedy answer, GRPO the mean of a group of samples (slide 41).
Here we run them on a task small enough to train in seconds on a CPU, with a reward that is **checked, not learned** (RLVR):

* **Task:** answer times-table questions such as `7x8=`. A rule-based verifier gives reward 1 for the right product, 0 otherwise.
* **Policy:** a tiny GRU language model that writes the answer one token (digit) at a time.
* **Starting point:** we first train it with SFT on answers from a **sloppy teacher** who often confuses neighbouring entries of the
  table. SFT copies the teacher's mistakes (slide 9: "bounded by the demonstrator"). Can RL with a verifier go beyond the teacher?

You will (1) compute GRPO advantages by hand, (2) measure pass@k and see why all-wrong groups give no gradient, (3) measure how much
each baseline reduces gradient noise, (4) train REINFORCE, ReMax and GRPO, and (5) compare pass@k before and after RL.

Runs on CPU in about 2–4 minutes. No downloads, no GPU.

In [ ]:
import math, random, copy, time
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
torch.manual_seed(0); random.seed(0); np.random.seed(0)

VOCAB = list("0123456789x=") + ["<e>"]                 # digits, 'x', '=', end-of-answer
stoi = {c: i for i, c in enumerate(VOCAB)}; EOS = stoi["<e>"]
PROMPTS = [(a, b) for a in range(1, 10) for b in range(1, 10)]     # 81 questions
def ptxt(a, b): return f"{a}x{b}="
PLEN = 4                                                # every prompt is 4 tokens long

def verify(p, answer):                                  # the rule-based reward of RLVR
    return 1.0 if answer == str(p[0] * p[1]) else 0.0

def teacher(a, b):                                      # a sloppy demonstrator, worse on big products
    if random.random() < 0.15 + 0.75 * (a * b / 81):
        return str(max(1, a * b + random.choice([-a, a, -b, b])))   # e.g. 7x8 -> 49 or 64
    return str(a * b)

print([f"{ptxt(a, b)}{teacher(a, b)}" for a, b in random.sample(PROMPTS, 8)])

## 0 · A tiny language model, trained by imitation (SFT)
The policy reads the prompt tokens and generates the answer tokens until `<e>`. The loss is on the answer tokens only, as in SFT
(slide 6). We train on 30 teacher answers per question.

In [ ]:
class Policy(nn.Module):
    def __init__(self, H=128):
        super().__init__()
        self.emb = nn.Embedding(len(VOCAB), 32); self.rnn = nn.GRU(32, H, batch_first=True); self.out = nn.Linear(H, len(VOCAB))
    def forward(self, x, h=None):
        y, h = self.rnn(self.emb(x), h); return self.out(y), h

def enc(s): return [stoi[c] for c in s]
def prompt_tensor(ps): return torch.tensor([enc(ptxt(a, b)) for a, b in ps])

def generate(model, ps, greedy=False):
    # sample (or greedily decode) one answer per prompt; returns strings and the answer tokens
    with torch.no_grad():
        lo, h = model(prompt_tensor(ps)); logit = lo[:, -1]
        out = torch.full((len(ps), 3), EOS); done = torch.zeros(len(ps), dtype=torch.bool)
        for t in range(3):
            nxt = logit.argmax(-1) if greedy else torch.multinomial(F.softmax(logit, -1), 1)[:, 0]
            nxt = torch.where(done, torch.full_like(nxt, EOS), nxt); out[:, t] = nxt; done |= nxt == EOS
            if done.all(): break
            lo, h = model(nxt[:, None], h); logit = lo[:, 0]
    strs = []
    for row in out.tolist():
        strs.append("".join(VOCAB[c] for c in (row[:row.index(EOS)] if EOS in row else row)))
    return strs, out

def logprob(model, ps, out):
    # log pi(answer | prompt) = sum of the answer tokens' log-probs (tokens after <e> are ignored)
    x = torch.cat([prompt_tensor(ps), out], 1)
    lo, _ = model(x[:, :-1])
    lp = F.log_softmax(lo[:, PLEN - 1:], -1).gather(-1, out[..., None])[..., 0]
    alive = torch.ones_like(lp)
    for t in range(1, out.shape[1]): alive[:, t] = alive[:, t - 1] * (out[:, t - 1] != EOS).float()
    return (lp * alive).sum(1)

def sft(model, n_per=30, epochs=6, lr=3e-3, bs=128):
    data = [(a, b, teacher(a, b)) for a, b in PROMPTS for _ in range(n_per)]
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for ep in range(epochs):
        random.shuffle(data)
        for i in range(0, len(data), bs):
            batch = data[i:i + bs]
            out = torch.full((len(batch), 3), EOS)
            for j, (a, b, y) in enumerate(batch): out[j, :len(y) + 1] = torch.tensor(enc(y) + [EOS])
            loss = -logprob(model, [(a, b) for a, b, _ in batch], out).mean()
            opt.zero_grad(); loss.backward(); opt.step()
    return data

def sample_acc(model, n=16):
    # per-question success rate p, estimated from n samples each
    ps = [p for p in PROMPTS for _ in range(n)]; s, _ = generate(model, ps)
    return np.array([verify(p, x) for p, x in zip(ps, s)]).reshape(len(PROMPTS), n).mean(1)

t0 = time.time()
base = Policy(); data = sft(base)
p_sft = sample_acc(base, 32)
greedy, _ = generate(base, PROMPTS, greedy=True)
print(f"SFT done in {time.time() - t0:.0f}s")
print(f"teacher accuracy            {np.mean([verify((a, b), y) for a, b, y in data]):.2f}")
print(f"SFT policy, sampled answers {p_sft.mean():.2f}")
print(f"SFT policy, greedy answers  {np.mean([verify(p, x) for p, x in zip(PROMPTS, greedy)]):.2f}")

Sampling from the SFT policy reproduces the teacher's mistakes (it is roughly as unreliable as the teacher, or worse). Greedy decoding
does better because the right answer is usually the *most likely* one; the model "knows" more than it reliably says. That gap is what
RL can exploit.

## 1 · GRPO advantages by hand
For each prompt GRPO samples a group of $G$ answers, scores them, and standardizes the rewards inside the group (slides 57–58):
$$A_i = \frac{r_i - \operatorname{mean}(r_1..r_G)}{\operatorname{std}(r_1..r_G)}.$$

In [ ]:
def grpo_adv(r, eps=1e-4, unbiased=False):
    r = torch.as_tensor(r, dtype=torch.float)
    return (r - r.mean(-1, keepdim=True)) / (r.std(-1, keepdim=True, unbiased=unbiased) + eps)

for r in [[1, 0, 1, 0], [1, 1, 1, 0], [0, 0, 0, 1], [1, 1, 1, 1], [0, 0, 0, 0]]:
    print(r, "->", grpo_adv(r).numpy().round(2), "   (unbiased std:", grpo_adv(r, unbiased=True).numpy().round(2), ")")

* `[1,0,1,0]` gives ±1, as on slide 58. Above the group average goes up, below goes down.
* A rare success (`[0,0,0,1]`) gets a **big** advantage; a rare failure gets a big negative one. Implementations differ in the
  standard deviation they use (population or unbiased), which only rescales the advantages.
* `[1,1,1,1]` and `[0,0,0,0]` give **all zeros**: no gradient at all from that prompt (slide 75).

## 2 · pass@k, and why hard prompts give no signal
If a question is solved with probability $p$ per sample, the chance that *at least one* of $k$ samples is right is pass@k. From $n$
samples with $c$ correct, the standard unbiased estimate (from the Codex paper) is $1 - \binom{n-c}{k}/\binom{n}{k}$.
A GRPO group of size $G$ is useless when it is all wrong, probability $(1-p)^G$, or all right, probability $p^G$.

In [ ]:
def pass_at_k(n, c, k):
    return 1.0 if n - c < k else 1.0 - math.comb(n - c, k) / math.comb(n, k)

def pass_curve(p_hat, n=32, ks=(1, 2, 4, 8, 16, 32)):
    cs = np.round(p_hat * n).astype(int)
    return [np.mean([pass_at_k(n, c, k) for c in cs]) for k in ks]

ks = (1, 2, 4, 8, 16, 32)
print("SFT policy pass@k:", {k: round(v, 3) for k, v in zip(ks, pass_curve(p_sft))})

G = 8
dead_wrong, dead_right = (1 - p_sft) ** G, p_sft ** G
print(f"\nwith G={G}: expected share of groups that are all wrong {dead_wrong.mean():.1%}, all right {dead_right.mean():.1%}")
hard = np.argsort(p_sft)[:5]
print("hardest questions:", [(ptxt(*PROMPTS[i]), round(p_sft[i], 2)) for i in hard])

# Slide 76: expected rollouts until the first useful (non-zero) gradient = 1/p + 1/(1-p) - 1
for p in [0.5, 0.1, 0.01]:
    sims = []
    for _ in range(2000):
        seen, n = set(), 0
        while len(seen) < 2: n += 1; seen.add(random.random() < p)
        sims.append(n)
    print(f"p={p:<5} formula {1 / p + 1 / (1 - p) - 1:7.1f}   simulation {np.mean(sims):7.1f}")

With $p = 0.01$ you need about 100 rollouts before a single useful gradient, but GRPO gives every prompt the same $G$. Hard prompts are
not hopeless, they are **under-sampled** (the Knapsack-RL view on slides 76–77).

## 3 · Baselines and gradient noise
RLHF rewards come from a reward model whose scores sit in different ranges for different prompts (slides 42 and 51). We imitate that by
adding a random **offset per prompt** to the 0/1 verifier reward. The offset carries no information about which answer is better, but
plain REINFORCE (no baseline) cannot tell. We estimate the policy gradient 30 times with each method and measure how noisy it is.

In [ ]:
offset = {p: 5.0 * np.random.randn() for p in PROMPTS}            # prompt-specific reward scale, like RM scores
def rm_style_reward(p, answer): return verify(p, answer) + offset[p]

def advantages(method, model, batch, r):
    # r: [P prompts, G samples]
    if method == "REINFORCE":  return r                                      # no baseline
    if method == "ReMax":                                                    # baseline = reward of the greedy answer
        g, _ = generate(model, batch, greedy=True)
        return r - torch.tensor([reward_fn(p, x) for p, x in zip(batch, g)])[:, None]
    if method == "GRPO":       return grpo_adv(r)                            # baseline = group mean, scaled by group std

def pg_loss(model, method, P=32, G=8):
    batch = random.sample(PROMPTS, P); ps = [p for p in batch for _ in range(G)]
    s, out = generate(model, ps)
    r = torch.tensor([reward_fn(p, x) for p, x in zip(ps, s)]).view(P, G)
    A = advantages(method, model, batch, r)
    return -(A.reshape(-1) * logprob(model, ps, out)).mean(), r

def grad_noise(method, reps=30):
    gs = []
    for _ in range(reps):
        base.zero_grad(); loss, _ = pg_loss(base, method, P=16); loss.backward()
        gs.append(torch.cat([q.grad.flatten() for q in base.parameters()]).clone())
    gs = torch.stack(gs); mu = gs.mean(0)
    return F.cosine_similarity(gs, mu[None]).mean().item()

reward_fn = rm_style_reward
for m in ["REINFORCE", "ReMax", "GRPO"]:
    print(f"{m:10s} mean cosine between one gradient estimate and the average direction: {grad_noise(m):.2f}")

A higher cosine means each single estimate points closer to the true direction, i.e. less noise. Without a baseline the per-prompt
offsets swamp the signal; subtracting the greedy reward (ReMax) or the group mean (GRPO) cancels them, because the baseline is
**prompt-specific** (slide 52).

## 4 · Train with REINFORCE, ReMax and GRPO
Now train each method for 100 steps (32 prompts × 8 samples per step) on the offset reward, and track the accuracy of *sampled*
answers and, for GRPO, the share of groups with no signal.

In [ ]:
def train_rl(method, steps=100, lr=2e-3, every=10):
    model = copy.deepcopy(base); opt = torch.optim.Adam(model.parameters(), lr=lr)
    hist, dead = [], []
    for step in range(steps + 1):
        if step % every == 0: hist.append(sample_acc(model, 8).mean())
        if step == steps: break
        loss, r = pg_loss(model, method)
        opt.zero_grad(); loss.backward(); opt.step()
        dead.append(((r == r[:, :1]).all(1)).float().mean().item())        # groups where every reward is equal
    return model, hist, dead

reward_fn = rm_style_reward
runs = {}
for m in ["REINFORCE", "ReMax", "GRPO"]:
    t0 = time.time(); runs[m] = train_rl(m); print(f"{m:10s} {time.time() - t0:4.0f}s   final sampled accuracy {runs[m][1][-1]:.2f}")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
for m, (_, hist, dead) in runs.items():
    ax[0].plot(range(0, 101, 10), hist, "o-", label=m)
ax[0].axhline(np.mean([verify((a, b), y) for a, b, y in data]), ls=":", c="gray", label="teacher")
ax[0].set_xlabel("RL step"); ax[0].set_ylabel("accuracy of sampled answers"); ax[0].legend(); ax[0].set_title("RL beyond the demonstrator")
ax[1].plot(np.convolve(runs["GRPO"][2], np.ones(5) / 5, mode="valid"))
ax[1].set_xlabel("RL step"); ax[1].set_ylabel("share of groups"); ax[1].set_title("GRPO: groups with zero advantage")
plt.show()

GRPO and ReMax pass the teacher they were imitating: the verifier rewards the right product even when the demonstrations got it wrong.
REINFORCE without a baseline is pushed around by the offsets. On the right, the share of zero-advantage groups **grows** as the policy
improves, because easy questions become all-correct: compute spent on them is wasted. DAPO's *dynamic sampling* drops such groups and
samples new prompts instead (slide 64).

## 5 · pass@k before and after RL

In [ ]:
p_grpo = sample_acc(runs["GRPO"][0], 32)
fig, ax = plt.subplots(figsize=(5.5, 3.6))
ax.plot(ks, pass_curve(p_sft), "o-", label="SFT"); ax.plot(ks, pass_curve(p_grpo), "s-", label="SFT + GRPO")
ax.set_xscale("log", base=2); ax.set_xlabel("k (samples)"); ax.set_ylabel("pass@k"); ax.legend(); ax.set_title("pass@k on the 81 questions")
plt.show()
print("SFT        :", [round(v, 3) for v in pass_curve(p_sft)])
print("SFT + GRPO :", [round(v, 3) for v in pass_curve(p_grpo)])

RL lifts **pass@1** a lot, but **pass@32** hardly moves (it may even drop a little, as RL narrows the distribution): with enough
samples the SFT model already found the right answer. Here RL
mainly makes the model *reliably* produce what it could already produce sometimes, the "RL amplifies what the base model has" theme of
slide 5. The same gap between pass@1 and pass@k is what repeated sampling and verifiers exploit at inference time (slide 122;
Notebook 3).

## Exercises
1. **Dr. GRPO** (slide 64) drops the division by the standard deviation. Change `advantages` to `r - r.mean(-1, keepdim=True)` and
   compare learning curves. Which groups lose the most weight?
2. **Dynamic sampling:** in `pg_loss`, drop groups whose rewards are all equal and count how many rollouts you saved over 100 steps.
3. **Budget allocation:** give each prompt a group size based on its current success rate (more samples where $p$ is small but not 0,
   in the spirit of Knapsack RL, slide 77). Does accuracy on the hardest five questions improve faster?
4. Make the teacher *always* wrong on `9x9` (edit `teacher`). Can RL still learn it? Look at pass@32 of the SFT model on that question
   and explain with section 2.

**Previous:** [Notebook 1 — reward models and DPO](https://colab.research.google.com/github/NLP-Course-CUHKSZ/NLP-course-cuhksz.github.io/blob/main/materials/tutorials/lecture06/lecture06_1_reward_models_and_dpo.ipynb) · **Next:** [Notebook 3 — test-time compute with a small LLM](https://colab.research.google.com/github/NLP-Course-CUHKSZ/NLP-course-cuhksz.github.io/blob/main/materials/tutorials/lecture06/lecture06_3_test_time_compute.ipynb)